# Imports

In [1]:
import os
import sys
import operator

# To define constants
from typing_extensions import Literal

# Langchain LLM model integration
from langchain_groq import ChatGroq

# To support older python versions
from typing_extensions import Optional, Annotated, Sequence

# Different Message types for defining chat model ip and op
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage, get_buffer_string

# Langgraph Entities to build and design control flow
from langgraph.graph import MessagesState, StateGraph, START, END
from langgraph.types import Command

# Persistence Layer - Checkpoints 
from langgraph.checkpoint.memory import InMemorySaver

# Chat History Management to append messages
from langgraph.graph.message import add_messages

# For runtime validation on state and output schemas

# Format Markdowns
from rich.markdown import Markdown

# Custom Imports
sys.path.append(os.path.abspath("../"))
from src.prompts import clarify_with_user_instructions, transform_messages_into_research_topic_prompt
from src.utils.util_functions import format_markdown_messages

# Load environment variables
from dotenv import load_dotenv
load_dotenv()

C:\SK7 Officials\langgraph-deep-research\langchain_env\Lib\site-packages\langchain_core\_api\deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1


True

# Defining States

In [2]:
class BriefingAgentInputState(MessagesState):
	"""Input state with only messages from user input."""
	pass

In [3]:
class BriefingAgentOutputState(MessagesState):
	"""
	Output state with the genenrated research brief
	"""
	# Research brief generated from user conversation history
	research_brief: str

# Scope Research Workflow

*User Clarification and Research Brief Generation*

This module implements the scoping phase of the research workflow, where we:
1. Assess if the user's request needs clarification
2. Generate a detailed research brief from the conversation

The workflow uses structured output to make deterministic decisions about
whether sufficient context exists to proceed with research.
"""

## Setting Up LLM

In [4]:
llm = ChatGroq(
    model="llama-3.3-70b-versatile", # better reasoning model
    temperature=0, # Factual response
    groq_api_key=os.getenv("GROQ_API_KEY"))  

## Parsing Utiity Functions

In [5]:
def parse_clarification_response(text: str):
    """
    Parse response from llm model to map with expected structured output
    """
    lines = text.lower().split("\n")

    result = {
        "need_clarification": False,
        "question": "",
        "verification": ""
    }

    for line in lines:
        if "need_clarification" in line:
            result["need_clarification"] = "true" in line

        elif line.startswith("question"):
            result["question"] = line.split(":", 1)[-1].strip()

        elif line.startswith("verification"):
            result["verification"] = line.split(":", 1)[-1].strip()

    return result

## Defining Nodes

In [6]:
def clarify_with_user(state: BriefingAgentInputState) -> Command[Literal["write_research_brief", "__end__"]]:
    """
    Determine if the user's request contains sufficient information to proceed with research.
    """

    # Invoke the model with clarification instructions and user conversations
    # get_buffer_string => list of messages to a string
    response = llm.invoke([HumanMessage(content=clarify_with_user_instructions.format(messages=get_buffer_string(messages=state.get("messages", []))))])

    # To match expected structured output
    parsed_response = parse_clarification_response(response.content)
    
    # Route based on clarification need
    # If clarification is needed, ask ; else generate research brief
    if parsed_response['need_clarification']:
        return Command(
            goto=END, 
            update={"messages": [AIMessage(content=parsed_response['question'])]} # Follow up question is added to AI Messge history
        )
    else:
        return Command(
            goto="write_research_brief", 
            update={"messages": [AIMessage(content=parsed_response['verification'])]} # Verification acknowledgement is added to AI Message history
        )

In [7]:
def write_research_brief(state: BriefingAgentInputState):
    """
    Transform the conversation history into a comprehensive research brief.
    """
    
    # Generate research brief from conversation history)
    research_brief = llm.invoke([HumanMessage(content=transform_messages_into_research_topic_prompt.format(
                                                                                            messages=get_buffer_string(messages=state.get("messages", []))))])
    
    # Update state with generated research brief and pass it to the supervisor
    return {
        "research_brief": research_brief.content,
    }

## Graph Construction

In [8]:
# Build the scoping workflow
scope_agent_builder = StateGraph(BriefingAgentOutputState, input_schema=BriefingAgentInputState)

# Add workflow nodes
scope_agent_builder.add_node("clarify_with_user", clarify_with_user)
scope_agent_builder.add_node("write_research_brief", write_research_brief)

# Add workflow edges
scope_agent_builder.add_edge(START, "clarify_with_user") # END if further clarification needed, else goto write_research_brief
scope_agent_builder.add_edge("write_research_brief", END)

# Compile the workflow
scope_agent = scope_agent_builder.compile(checkpointer=InMemorySaver())

## Sample Conversation

In [9]:
# Thread to store the checkpoints
thread = {"configurable": {"thread_id": "1"}}

result = scope_agent.invoke({"messages": [HumanMessage(content="I want to research the best chennai restaurants.")]}, config=thread)

In [10]:
format_markdown_messages(result['messages'])

### 🧑 Human
I want to research the best chennai restaurants.

### 🤖 AI
what type of cuisine are you looking for in chennai restaurants?



In [11]:
result = scope_agent.invoke({"messages": [HumanMessage(content="Focus on Indian cuisine")]}, config=thread)

In [12]:
format_markdown_messages(result['messages'])

### 🧑 Human
I want to research the best chennai restaurants.

### 🤖 AI
what type of cuisine are you looking for in chennai restaurants?

### 🧑 Human
Focus on Indian cuisine

### 🤖 AI
i have sufficient information to proceed. you're looking for the best restaurants in chennai, specifically focusing on indian cuisine. i will now begin the research process to provide you with the top recommendations.



In [13]:
Markdown(result["research_brief"])

What are the top-rated Indian cuisine restaurants in Chennai, based on reliable sources such as official reviews,  
culinary awards, and verified reports, that I can consider for dining, with other factors like location, price     
range, and amenities remaining open for consideration?